# 🎬 Manga Recap AI Studio (Google Colab T4 + Gradio UI)
> **Tự động hóa sản xuất Video Tóm tắt / Review Truyện tranh Kịch tính (Quán Khuya Style)**  
> Mặc định môi trường: **Python 3 + GPU NVIDIA Tesla T4 (16GB VRAM)**.  
> Tích hợp **Giao diện Gradio Web UI** trực quan, hỗ trợ xem trước và tương tác dễ dàng.

### ⚡ Bước 1: Kiểm tra GPU T4 & Kết nối Google Drive (Tùy chọn)
- **Lưu ý:** Bạn có thể kết nối Google Drive để lưu trữ video thành phẩm vĩnh viễn.
- **Nếu KHÔNG mount Google Drive:** Hệ thống sẽ tự động lưu video an toàn vào thư mục `/content/workspace` trên Colab.

In [1]:
# 1. Kiểm tra cấu hình GPU Tesla T4
!nvidia-smi

# 2. Kết nối Google Drive (Nếu không mount, hệ thống tự động lưu vào /content/workspace)
import os
from pathlib import Path

USE_DRIVE = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = Path('/content/drive/MyDrive/MangaRecap')
    drive_dir.mkdir(parents=True, exist_ok=True)
    USE_DRIVE = True
    print("✅ Đã kết nối Google Drive! Kết quả sẽ tự động sao lưu vào /content/drive/MyDrive/MangaRecap")
except Exception as e:
    print("ℹ️ Chưa kết nối Google Drive. Toàn bộ file sẽ được lưu cục bộ tại /content/workspace trên Colab.")

Sun Oct  4 05:02:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 📦 Bước 2: Cài đặt Toàn Bộ Thư Viện Cần Thiết
Cài đặt Gradio, VieNeu-TTS, Faster-Whisper, Ultralytics, Google GenAI SDK...

In [2]:
!git clone https://github.com/tuna1710/REVIEWTRUYENTRANH.git
%cd REVIEWTRUYENTRANH

Cloning into 'REVIEWTRUYENTRANH'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 64 (delta 21), reused 52 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 545.57 KiB | 24.80 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/REVIEWTRUYENTRANH


In [3]:
# Cài đặt các thư viện trong requirements.txt
!pip install -q -r requirements.txt
print("✅ Đã cài đặt xong toàn bộ môi trường!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 467.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 941.3/941.3 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.9/334.9 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7

### 🌟 Bước 3: Khởi Chạy Giao Diện Gradio Web UI
Chạy lệnh bên dưới để mở giao diện web trực quan.  
Gradio sẽ cung cấp:  
1. **Giao diện hiển thị trực tiếp bên trong Colab.**  
2. **Đường link Public (`https://xxxx.gradio.live`)** để bạn có thể mở toàn màn hình trên trình duyệt hoặc điện thoại.

In [ ]:
# Khởi chạy Gradio Web App
!python app.py

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
/content/REVIEWTRUYENTRANH/app.py:211: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme, title="Manga Recap AI Studio") as demo:
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://61baa24b201128b028.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
📁 Thư mục lưu trữ: /content/workspace
📦 Đang giải nén/trích xuất

### 🎬 Tùy chọn 2: Chạy trực tiếp qua dòng lệnh CLI với truyện tranh mẫu thực tế
Hệ thống đã tích hợp sẵn tập truyện kinh dị mẫu `assets/mieruko_chan_ch01.cbz` (Mieruko-chan Ch.1). Bạn có thể chạy ngay để xuất video dọc 9:16 (Shorts/TikTok) hoặc 16:9:

In [ ]:
# 1. Chạy test case tự động toàn bộ pipeline:
!python -m unittest discover tests

# 2. Dựng video hoàn chỉnh từ truyện mẫu Mieruko-chan:
!python main.py \
  --input "assets/mieruko_chan_ch01.cbz" \
  --synopsis "Cô nữ sinh nhìn thấy quái vật rùng rợn và phải giả vờ như không thấy gì..." \
  --voice "Thiện Minh" \
  --aspect-ratio "9:16" \
  --capcut
